# Cross-Run Drift Detection

How do pipeline results change between two benchmark runs (different reference
packs, systems, or timestamps)?

This notebook loads two `benchmark_pack.json` files, runs the lake comparison
engine, and visualizes gate deltas and metric distribution shifts. Adjust the
configuration cell below to point at your benchmark packs.

In [ ]:
# ============================================================
# CONFIGURATION — edit these variables before running
# ============================================================
from pathlib import Path
from materials_discovery.common.io import workspace_root

WORKSPACE = workspace_root()

# Paths to two benchmark_pack.json files to compare.
# Defaults point at the Phase 4 benchmark outputs.
PACK_A_PATH = WORKSPACE / "data" / "benchmarks" / "al_cu_fe_benchmark.json"
PACK_B_PATH = WORKSPACE / "data" / "benchmarks" / "sc_zn_benchmark.json"

print(f"Pack A: {PACK_A_PATH}")
print(f"  exists: {PACK_A_PATH.exists()}")
print(f"Pack B: {PACK_B_PATH}")
print(f"  exists: {PACK_B_PATH.exists()}")

In [ ]:
# ============================================================
# Load benchmark packs as LaneSnapshots
# ============================================================
import warnings
from materials_discovery.lake.compare import LaneSnapshot, compare_benchmark_packs

_missing = [p for p in [PACK_A_PATH, PACK_B_PATH] if not p.exists()]
if _missing:
    print("[INFO] Benchmark pack(s) not found:")
    for p in _missing:
        print(f"  {p}")
    print("Run the benchmark suite first: bash scripts/run_reference_aware_benchmarks.sh")
    snap_a = snap_b = comparison = None
else:
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        snap_a = LaneSnapshot.from_benchmark_pack(PACK_A_PATH)
        snap_b = LaneSnapshot.from_benchmark_pack(PACK_B_PATH)
        comparison = compare_benchmark_packs(PACK_A_PATH, PACK_B_PATH)
    if caught:
        for w in caught:
            print(f"[WARN] {w.message}")
    print(f"Lane A: {snap_a.system} (lane_id={snap_a.lane_id})")
    print(f"  gates: {list(snap_a.gate_results.keys())}")
    print(f"  metrics: {list(snap_a.metric_distributions.keys())}")
    print(f"Lane B: {snap_b.system} (lane_id={snap_b.lane_id})")
    print(f"  gates: {list(snap_b.gate_results.keys())}")
    print(f"  metrics: {list(snap_b.metric_distributions.keys())}")

In [ ]:
# ============================================================
# Grouped bar chart: gate results comparison
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

if comparison is not None and comparison.gate_deltas:
    gates = [gd.gate_name for gd in comparison.gate_deltas]
    a_vals = [1 if gd.lane_a else 0 for gd in comparison.gate_deltas]
    b_vals = [1 if gd.lane_b else 0 for gd in comparison.gate_deltas]
    statuses = [gd.status for gd in comparison.gate_deltas]

    x = np.arange(len(gates))
    width = 0.35

    fig, ax = plt.subplots(figsize=(max(8, len(gates) * 1.2), 4))
    bars_a = ax.bar(x - width / 2, a_vals, width, label=snap_a.system, color="#3498db")
    bars_b = ax.bar(x + width / 2, b_vals, width, label=snap_b.system, color="#e67e22")

    # Highlight regressions
    for i, status in enumerate(statuses):
        if status == "regression":
            ax.annotate("REGR", xy=(x[i] + width / 2, b_vals[i] + 0.05), ha="center", fontsize=8, color="red")
        elif status == "improvement":
            ax.annotate("IMPR", xy=(x[i] + width / 2, b_vals[i] + 0.05), ha="center", fontsize=8, color="green")

    ax.set_xticks(x)
    ax.set_xticklabels(gates, rotation=30, ha="right")
    ax.set_yticks([0, 1])
    ax.set_yticklabels(["FAIL", "PASS"])
    ax.set_title("Gate Results Comparison")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()
elif comparison is not None:
    print("No gate deltas available (packs may have no release_gate entries).")
else:
    print("No comparison data — run the benchmark suite first.")

In [ ]:
# ============================================================
# Side-by-side bar charts for key metric distributions
# ============================================================
if comparison is not None and comparison.metric_deltas:
    metrics_with_data = [
        md for md in comparison.metric_deltas
        if md.lane_a is not None or md.lane_b is not None
    ]

    if not metrics_with_data:
        print("No per-entry metric distributions available.")
        print("Tip: ensure the report file is accessible from stage_manifest_paths.")
    else:
        n = len(metrics_with_data)
        fig, axes = plt.subplots(1, n, figsize=(max(6, n * 3), 4), sharey=False)
        if n == 1:
            axes = [axes]

        for ax, md in zip(axes, metrics_with_data):
            means = [
                md.lane_a.mean if md.lane_a else 0,
                md.lane_b.mean if md.lane_b else 0,
            ]
            stds = [
                md.lane_a.std if md.lane_a else 0,
                md.lane_b.std if md.lane_b else 0,
            ]
            labels = [snap_a.system, snap_b.system]
            colors = ["#3498db", "#e67e22"]
            bars = ax.bar(labels, means, color=colors, alpha=0.8)
            ax.errorbar(labels, means, yerr=stds, fmt="none", color="black", capsize=5)
            ax.set_title(md.metric_name, fontsize=9)
            ax.set_ylabel("mean")
            ax.grid(axis="y", alpha=0.3)

        plt.suptitle("Metric Distribution Comparison (mean ± std)", fontsize=12)
        plt.tight_layout()
        plt.show()
else:
    print("No metric data available.")

In [ ]:
# ============================================================
# Delta table: metric_name | mean_a | mean_b | delta | interpretation
# ============================================================
def interpret_delta(metric: str, delta: float | None) -> str:
    """Simple heuristic interpretation of a metric delta."""
    if delta is None:
        return "N/A"
    if abs(delta) < 1e-6:
        return "no change"
    # For score-like metrics: lower is better
    lower_is_better = {"hifi_score", "ood_score", "delta_e_proxy_hull_ev_per_atom", "uncertainty_ev_per_atom"}
    higher_is_better = {"stability_probability", "xrd_confidence", "xrd_distinctiveness", "md_stability_score"}
    if metric in lower_is_better:
        return "B better" if delta < 0 else "A better"
    if metric in higher_is_better:
        return "B better" if delta > 0 else "A better"
    return "shifted"


if comparison is not None and comparison.metric_deltas:
    print(f"{'Metric':<40} {'Mean A':>10} {'Mean B':>10} {'Delta':>10} {'Interpretation'}")
    print("-" * 82)
    for md in comparison.metric_deltas:
        mean_a = f"{md.lane_a.mean:.4f}" if md.lane_a else "N/A"
        mean_b = f"{md.lane_b.mean:.4f}" if md.lane_b else "N/A"
        delta_str = f"{md.delta_mean:+.4f}" if md.delta_mean is not None else "N/A"
        interp = interpret_delta(md.metric_name, md.delta_mean)
        print(f"{md.metric_name:<40} {mean_a:>10} {mean_b:>10} {delta_str:>10} {interp}")
else:
    print("No comparison data available.")